In [1]:
import networkx as nx
import pandas as pd


# ΔΕΔΟΜΕΝΑ (από την άσκηση)
dur = {
    "A": 4, "B": 9, "C": 7, "D": 5,
    "E": 6, "F": 4, "G": 3, "H": 1
}

pred = {
    "A": [],
    "B": ["A"],
    "C": ["B"],
    "D": ["A"],
    "E": ["D"],
    "F": ["C", "E"],
    "G": ["F"],
    "H": ["G"]
}

# AON ΓΡΑΦΗΜΑ

G = nx.DiGraph()
for a, d in dur.items():
    G.add_node(a, duration=d)
for a, preds in pred.items():
    for p in preds:
        G.add_edge(p, a)

order = list(nx.topological_sort(G))

# FORWARD PASS: ES, EF

ES, EF = {}, {}
for a in order:
    ES[a] = max((EF[p] for p in pred[a]), default=0)
    EF[a] = ES[a] + dur[a]

project_duration = max(EF.values())

# BACKWARD PASS: LS, LF

LS, LF = {}, {}
for a in reversed(order):
    succ = list(G.successors(a))
    if not succ:
        LF[a] = project_duration
    else:
        LF[a] = min(LS[s] for s in succ)
    LS[a] = LF[a] - dur[a]

# Slack

Slack = {a: LS[a] - ES[a] for a in order}

# ΑΠΟΤΕΛΕΣΜΑΤΑ

df = pd.DataFrame({
    "Duration": [dur[a] for a in order],
    "ES": [ES[a] for a in order],
    "EF": [EF[a] for a in order],
    "LS": [LS[a] for a in order],
    "LF": [LF[a] for a in order],
    "Slack": [Slack[a] for a in order],
}, index=order)

critical_acts = [a for a in order if Slack[a] == 0]
critical_path = nx.dag_longest_path(G, weight="duration")

print("Συνολική διάρκεια έργου:", project_duration, "ημέρες")
print("Κρίσιμες δραστηριότητες (Slack=0):", " -> ".join(critical_acts))
print("Critical Path (longest path):", " -> ".join(critical_path))

df

# CRASHING (C και F) + Κόστος–Όφελος

crash_limits = {"C": 2, "F": 2}
crash_cost_per_day = {"C": 900, "F": 1100}
benefit_per_day_early = 1700

base_duration = project_duration  # η διάρκεια του έργου (28)

results = []

for c in range(crash_limits["C"] + 1):      
    for f in range(crash_limits["F"] + 1):  

        # 1) Νέες διάρκειες μετά το crash
        dur_new = dur.copy()
        dur_new["C"] = max(0, dur_new["C"] - c)
        dur_new["F"] = max(0, dur_new["F"] - f)

        # 2) Ξανατρέχουμε μόνο το forward pass για να βρούμε νέα συνολική διάρκεια
        G_new = nx.DiGraph()
        for a, d in dur_new.items():
            G_new.add_node(a, duration=d)
        for a, preds in pred.items():
            for p in preds:
                G_new.add_edge(p, a)

        order_new = list(nx.topological_sort(G_new))

        ES2, EF2 = {}, {}
        for a in order_new:
            ES2[a] = max((EF2[p] for p in pred[a]), default=0)
            EF2[a] = ES2[a] + dur_new[a]

        new_duration = max(EF2.values())

        # 3) Μείωση, κόστος, όφελος
        reduction = base_duration - new_duration #Μείωση Χρόνου
        cost = c * crash_cost_per_day["C"] + f * crash_cost_per_day["F"]
        benefit = reduction * benefit_per_day_early
        profit = benefit - cost 

        results.append({
            "C_crash_days": c,
            "F_crash_days": f,
            "New_Duration": new_duration,
            "Reduction": reduction,
            "Crash_Cost": cost,
            "Benefit": benefit,
            "Profit": profit
        })

crash_df = pd.DataFrame(results).sort_values(["Reduction", "Crash_Cost"], ascending=[False, True])
crash_df = crash_df[crash_df["Reduction"] <= 3]



print("\n=== Πίνακας όλων των επιλογών crashing ===")
crash_df
display(crash_df)
best_rows = []
for t in [1, 2, 3]:
    best = (
        crash_df[crash_df["Reduction"] >= t]
        .sort_values(["Crash_Cost", "New_Duration"])
        .iloc[0]
    )
    best_rows.append(best)

best_df = pd.DataFrame(best_rows)

print("=== Καλύτερο σενάριο (ελάχιστο κόστος) για μείωση 1, 2, 3 ημερών ===")
display(best_df)



Συνολική διάρκεια έργου: 28 ημέρες
Κρίσιμες δραστηριότητες (Slack=0): A -> B -> C -> F -> G -> H
Critical Path (longest path): A -> B -> C -> F -> G -> H

=== Πίνακας όλων των επιλογών crashing ===


,C_crash_days,F_crash_days,New_Duration,Reduction,Crash_Cost,Benefit,Profit
7,2,1,25,3,2900,5100,2200
5,1,2,25,3,3100,5100,2000
6,2,0,26,2,1800,3400,1600
4,1,1,26,2,2000,3400,1400
2,0,2,26,2,2200,3400,1200
3,1,0,27,1,900,1700,800
1,0,1,27,1,1100,1700,600
0,0,0,28,0,0,0,0


=== Καλύτερο σενάριο (ελάχιστο κόστος) για μείωση 1, 2, 3 ημερών ===


,C_crash_days,F_crash_days,New_Duration,Reduction,Crash_Cost,Benefit,Profit
3,1,0,27,1,900,1700,800
6,2,0,26,2,1800,3400,1600
7,2,1,25,3,2900,5100,2200
